# esa-lookup (self-contained notebook)

Mass-lookup SAP data into an Excel workbook via transaction **ZTBV**, plant
**ESA1**. Two workflows:

- **TO** (3 steps): read col K → LTAP → write ABLAD to M; read N+O →
  Z50CFG_ENG_CRNT → write OBJNR/DISP_MATNR/DISP_QTY to C/D/E, QMNUM to A,
  composite key to P; read C → Z50CFG_ENG_VALD → write
  Z_SECTION/Z_MODULE/DESCRIPT/SALES_ORDER to F..I.
- **NOTIF** (2 steps): read A → Z50CFG_ENG_CRNT → write to C/D/E; read C →
  Z50CFG_ENG_VALD → write F..I + LID to J.

## Prerequisites

- Windows + **SAP GUI for Windows** installed (Java client not supported)
- **SAP GUI Scripting** enabled (Options → Accessibility & Scripting)
- **Microsoft Excel** desktop installed
- Python 3.10+ with `pandas`, `openpyxl`, `pywin32`:
  ```powershell
  pip install pandas openpyxl pywin32
  ```

## How to run

1. Log into SAP GUI (any session; script attaches to the first one).
2. Open the target Excel file in Excel.
3. Edit the **Configure your run** cell below (set `EXCEL_PATH` and `WORKFLOW`).
4. Cell → Run All.

## What this notebook contains

Everything from the multi-file `esa-lookup` project inlined into one file,
so the notebook can be handed to someone who does not have the repo.
Business logic is byte-identical to `pipeline.py` on `main` after the
notebook-parity fixes; the tkinter GUI is replaced by simple `print()` for
progress output.


## Imports and constants


In [ ]:
from __future__ import annotations

import contextlib
import os
import re
import sys
import tempfile
import threading
import time
import traceback
from dataclasses import dataclass, field
from decimal import Decimal, InvalidOperation

import pandas as pd
import pythoncom
import win32com.client
from win32com.client import constants  # noqa: F401 (loaded lazily by pywin32)


PLANT = "ESA1"
TRANSACTION = "ZTBV"

XL_UP = -4162
XL_CALC_MANUAL = -4135
XL_CALC_AUTOMATIC = -4105


class Cancelled(Exception):
    """Raised inside a worker step when the user has pressed Stop."""


class SapError(RuntimeError):
    pass


class ExcelError(RuntimeError):
    pass


## Key normalization

Applied identically to both the Excel side and the SAP side, so it can
only add matches, never subtract. See the `normalize_key` docstring for the
exact rules.


In [ ]:
# Only expand scientific notation when the WHOLE string matches sci-notation;
# an object number like "1E2000" would otherwise mangle to "100...".
_SCI_NOTATION_RE = re.compile(r"^-?\d+(\.\d+)?[eE][+-]?\d+$")

# Rows whose primary key cell equals one of these (case-insensitive) are
# not sent to SAP as filter values, matching the notebook's paste-side
# guard: `if v != "" and v.upper() not in {"NOT FOUND","NOTFOUND"}`.
_SKIP_KEY_MARKERS = frozenset({"NOT FOUND", "NOTFOUND"})


def _clean_cell(value) -> str:
    """Turn a raw Excel/pandas value into a stripped string. Treats None,
    NaN, and pywintypes cell-error ints (#N/A / #REF! / #VALUE! come back
    as large-negative int codes) as empty.
    """
    if value is None:
        return ""
    if isinstance(value, float):
        # NaN check without importing math.isnan
        if value != value:
            return ""
    if isinstance(value, int) and not isinstance(value, bool) and value < -2_000_000_000:
        return ""
    return str(value).strip()


def _canonicalize_number_shape(txt: str) -> str:
    for ch in (" ", "\t", chr(160), "'", ","):
        txt = txt.replace(ch, "")
    if _SCI_NOTATION_RE.match(txt):
        try:
            txt = format(Decimal(txt), "f")
        except InvalidOperation:
            pass
    if "." in txt:
        left, right = txt.split(".", 1)
        if right == "" or set(right) <= {"0"}:
            txt = left
    return txt


def normalize_key(value) -> str:
    """Aggressively normalize an Excel value so it matches SAP's key form:
    drop None/NaN/cell-errors, strip whitespace/NBSP/apostrophes/commas,
    collapse full-string scientific notation, drop trailing '.0'/'.00'/'.',
    drop leading zeros (keep at least one char)."""
    txt = _clean_cell(value)
    if not txt:
        return ""
    txt = _canonicalize_number_shape(txt)
    while len(txt) > 1 and txt.startswith("0"):
        txt = txt[1:]
    return txt


def clean_numeric_for_sap(value) -> str:
    """Like normalize_key but preserves leading zeros -- used when pasting
    numeric identifiers into SAP (SAP itself canonicalizes them)."""
    txt = _clean_cell(value)
    if not txt:
        return ""
    return _canonicalize_number_shape(txt)


def _is_skip_key_cell(value) -> bool:
    """True when this Excel key cell should NOT be pasted into SAP's filter
    dialog (blank, or one of the 'already resolved' markers). Skipped rows
    are still processed by the write-back loop as non-matches (notebook
    parity: the write loop iterates every row and the else-branch fires for
    any composite key not in the SAP result set)."""
    txt = _clean_cell(value)
    if not txt:
        return True
    return txt.upper() in _SKIP_KEY_MARKERS


## Workflow definitions

Three different write behaviors are modeled per column:

| Behavior                                          | Where it appears                    |
|---------------------------------------------------|-------------------------------------|
| write-on-match + clear-on-nonmatch                | all main outputs; LID on NOTIF-2    |
| write-on-match + preserve-on-nonmatch             | A on TO-2 (QMNUM)                   |
| preserve-on-match + clear-on-nonmatch             | J on TO-3                           |

`match_key_column` writes an audit-trail composite key on every non-skip
row (used by TO-2 for column P).


In [ ]:
@dataclass
class ExtraOutput:
    """A per-step output column that lives outside the main output block.

    `sap_col=None` → no SAP source; column is cleared on non-match, preserved
    on match (e.g. TO-3 col J).

    `preserve_on_nonmatch=True` → do not touch this cell on non-match
    (e.g. TO-2 col A per notebook comment "Do not touch Column A if there
    is no match").

    Skip rows (blank / "NOT FOUND" primary key) are treated identically to
    non-match rows for extras, matching notebook write-loop semantics.
    """
    excel_col: int
    sap_col: str | None = None
    preserve_on_nonmatch: bool = False


@dataclass
class LookupStep:
    name: str
    sap_table: str
    push_button_field: str
    key_columns: list[int]
    key_joiner: str = "|"
    sap_key_columns: list[str] = field(default_factory=list)
    sap_output_columns: list[str] = field(default_factory=list)
    excel_output_columns: list[int] = field(default_factory=list)
    extras: list[ExtraOutput] = field(default_factory=list)
    # Optional: 1-based column that receives the composite key as an audit
    # trail (TO-2 writes this to column P).
    match_key_column: int | None = None
    match_key_header: str = "Excel Match Key Used"


WORKFLOWS = {
    "TO": [
        LookupStep(
            name="LTAP -> Unloading Point",
            sap_table="LTAP",
            push_button_field="TO_NUMBER",
            key_columns=[11],                     # K
            sap_key_columns=["TANUM"],
            sap_output_columns=["ABLAD"],
            excel_output_columns=[13],            # M
        ),
        LookupStep(
            name="Z50CFG_ENG_CRNT (Reservation) -> QMNUM/OBJNR/DISP",
            sap_table="Z50CFG_ENG_CRNT",
            push_button_field="RSNUM",
            key_columns=[14, 15],                 # N | O
            sap_key_columns=["RSNUM", "RSPOS"],
            sap_output_columns=["OBJNR", "DISP_MATNR", "DISP_QTY"],
            excel_output_columns=[3, 4, 5],       # C, D, E
            extras=[ExtraOutput(excel_col=1, sap_col="QMNUM", preserve_on_nonmatch=True)],
            match_key_column=16,                  # P
        ),
        LookupStep(
            name="Z50CFG_ENG_VALD -> Section/Module/Description/SalesDoc",
            sap_table="Z50CFG_ENG_VALD",
            push_button_field="OBJNR",
            key_columns=[3],                      # C
            sap_key_columns=["OBJNR"],
            sap_output_columns=["Z_SECTION", "Z_MODULE", "DESCRIPT", "SALES_ORDER"],
            excel_output_columns=[6, 7, 8, 9],    # F..I
            extras=[ExtraOutput(excel_col=10)],   # J: preserve on match, clear on non-match
        ),
    ],
    "NOTIF": [
        LookupStep(
            name="Z50CFG_ENG_CRNT (Notification) -> OBJNR/DISP_MATNR/DISP_QTY",
            sap_table="Z50CFG_ENG_CRNT",
            push_button_field="QMNUM",
            key_columns=[1],                      # A
            sap_key_columns=["QMNUM"],
            sap_output_columns=["OBJNR", "DISP_MATNR", "DISP_QTY"],
            excel_output_columns=[3, 4, 5],       # C, D, E
        ),
        LookupStep(
            name="Z50CFG_ENG_VALD -> Section/Module/Description/SalesDoc/LID",
            sap_table="Z50CFG_ENG_VALD",
            push_button_field="OBJNR",
            key_columns=[3],                      # C
            sap_key_columns=["OBJNR"],
            sap_output_columns=["Z_SECTION", "Z_MODULE", "DESCRIPT", "SALES_ORDER"],
            excel_output_columns=[6, 7, 8, 9],    # F..I
            # LID -> J via extras so main ClearContents range matches
            # notebook (F..I only), preserving user data in J below last_row.
            extras=[ExtraOutput(excel_col=10, sap_col="LID", preserve_on_nonmatch=False)],
        ),
    ],
}


## SAP column aliases

ALV exports use the column TITLE, which is often a short description rather
than the technical field name. Try the technical name first, then aliases.
Extend per-site as needed.


In [ ]:
SAP_COLUMN_ALIASES: dict[str, list[str]] = {
    "TANUM":       ["TANUM", "TO Number", "Transfer Order", "TrfOrd", "TrfOrdNo"],
    "ABLAD":       ["ABLAD", "Unloading Point", "UnloadPt"],
    "QMNUM":       ["QMNUM", "Notification", "Notification No", "Notification Number"],
    "RSNUM":       ["RSNUM", "Reservation", "Reservation No", "Reservation Number", "Res.Number"],
    "RSPOS":       ["RSPOS", "Item", "Item No", "Item Number", "Res.Item"],
    "OBJNR":       ["OBJNR", "Object Number", "Obj.Number", "Object No"],
    "DISP_MATNR":  ["DISP_MATNR", "Disp Material", "Disposition Material", "Disp.Material"],
    "DISP_QTY":    ["DISP_QTY", "Disp Qty", "Disposition Qty", "Disposition Quantity", "Disp.Qty"],
    "Z_SECTION":   ["Z_SECTION", "Section"],
    "Z_MODULE":    ["Z_MODULE", "Module"],
    "DESCRIPT":    ["DESCRIPT", "Description", "Descr."],
    "SALES_ORDER": ["SALES_ORDER", "Sales Order", "Sales Doc.", "Sales Doc", "Sales Doc. No."],
    "LID":         ["LID"],
}


def _resolve_column(df: pd.DataFrame, canonical: str) -> str | None:
    for name in SAP_COLUMN_ALIASES.get(canonical, [canonical]):
        if name in df.columns:
            return name
    return None


## Excel COM helpers

Attach to a running Excel instance (or open the workbook), bulk read/write
ranges, and stage values on the OS clipboard for SAP paste.


In [ ]:
@dataclass
class ExcelCtx:
    app: object
    book: object
    sheet: object


def excel_attach(path: str) -> ExcelCtx:
    """Return an ExcelCtx for `path`. If Excel already has the file open,
    reuse it; otherwise start Excel and open it read/write."""
    if not os.path.exists(path):
        raise ExcelError(f"Excel file not found:\n{path}")

    abs_path = os.path.abspath(path).lower()

    try:
        app = win32com.client.GetActiveObject("Excel.Application")
    except Exception:
        app = win32com.client.Dispatch("Excel.Application")
        app.Visible = True

    # OneDrive/SharePoint-hosted workbooks report FullName as an https:// URL.
    # Match by URL-tail filename in that case.
    target_basename = os.path.basename(path).lower()
    book = None
    for i in range(1, app.Workbooks.Count + 1):
        b = app.Workbooks(i)
        try:
            fn = str(b.FullName)
        except Exception:
            continue
        fn_lower = fn.lower()
        if fn_lower.startswith(("http:", "https:")):
            tail = fn.rsplit("/", 1)[-1].lower()
            if tail == target_basename:
                book = b
                break
        else:
            try:
                if os.path.abspath(fn).lower() == abs_path:
                    book = b
                    break
            except Exception:
                continue

    if book is None:
        book = app.Workbooks.Open(os.path.abspath(path), 0, False)

    if book.ReadOnly:
        raise ExcelError(
            "Workbook is open Read-Only. Close it in Excel or check "
            "OneDrive/SharePoint permissions, then retry.\n" + path
        )

    return ExcelCtx(app=app, book=book, sheet=book.Worksheets(1))


def last_row_in_column(sheet, col_index: int) -> int:
    return int(sheet.Cells(sheet.Rows.Count, col_index).End(XL_UP).Row)


def read_range_2d(sheet, range_str: str) -> list[list]:
    raw = sheet.Range(range_str).Value
    if raw is None:
        return []
    if not isinstance(raw, tuple):
        return [[raw]]
    if raw and not isinstance(raw[0], tuple):
        return [list(raw)]
    return [list(r) for r in raw]


def write_range_2d(sheet, range_str: str, values_2d: list[list]) -> None:
    """Bulk write. Pre-checks for merged cells so a mid-write failure can
    never leave the sheet in a half-updated state."""
    if not values_2d:
        return
    rng = sheet.Range(range_str)
    try:
        merged = bool(rng.MergeCells)
    except Exception:
        # MergeCells returns None (not a bool) for a mixed-merge range.
        merged = True
    if merged:
        raise ExcelError(
            f"Cannot write to range {range_str}: it contains merged cells. "
            f"Unmerge those columns in Excel and re-run."
        )
    payload = tuple(tuple(row) for row in values_2d)
    rng.Value = payload


def clear_range(sheet, range_str: str) -> None:
    sheet.Range(range_str).ClearContents()


def set_column_format_text(sheet, columns: str) -> None:
    sheet.Range(f"{columns}").NumberFormat = "@"


@contextlib.contextmanager
def bulk_write(app):
    prev_updating = app.ScreenUpdating
    prev_events = app.EnableEvents
    prev_calc = app.Calculation
    try:
        app.ScreenUpdating = False
        app.EnableEvents = False
        app.Calculation = XL_CALC_MANUAL
        yield
    finally:
        app.Calculation = prev_calc
        app.EnableEvents = prev_events
        app.ScreenUpdating = prev_updating


def stage_values_on_clipboard(app, values: list[str]) -> object:
    """Create a scratch workbook, dump `values` into Column A as text,
    Copy() them onto the OS clipboard, return the scratch workbook."""
    scratch = app.Workbooks.Add()
    ws = scratch.Worksheets(1)
    ws.Columns("A").NumberFormat = "@"
    if values:
        payload = tuple(("" if v is None else str(v),) for v in values)
        ws.Range(f"A1:A{len(values)}").Value = payload
        ws.Range(f"A1:A{len(values)}").Copy()
    return scratch


def close_scratch(scratch) -> None:
    try:
        scratch.Close(SaveChanges=False)
    except Exception:
        pass


def excel_save(book) -> None:
    try:
        book.Save()
    except Exception as e:
        raise ExcelError(
            f"Excel refused to save the workbook: {e}. Data is written but "
            f"unsaved -- press Ctrl+S in Excel to persist it."
        ) from e


## SAP GUI scripting helpers


In [ ]:
@dataclass
class SapSession:
    session: object

    def find(self, oid):
        return self.session.findById(oid)


def sap_attach() -> SapSession:
    try:
        sap_gui = win32com.client.GetObject("SAPGUI")
    except Exception as e:
        raise SapError(
            "Cannot reach SAP GUI. Log into SAP GUI first and confirm "
            "'Enable scripting' is on (Options -> Accessibility & Scripting)."
        ) from e
    try:
        app = sap_gui.GetScriptingEngine
        conn = app.Children(0)
        sess = conn.Children(0)
    except Exception as e:
        raise SapError(
            "SAP GUI is running but no active session was found. Open a "
            "connection and log in, then retry."
        ) from e
    return SapSession(sess)


def close_lingering_modals(s: SapSession, log=None) -> int:
    """Close any wnd[1..N] popups left over from a previous run."""
    closed = 0
    for i in range(9, 0, -1):
        try:
            title = ""
            try:
                title = str(s.find(f"wnd[{i}]").Text or "")
            except Exception:
                pass
            s.find(f"wnd[{i}]").close()
            closed += 1
            if log:
                log(f"SAP: closed leftover modal wnd[{i}]" +
                    (f" (title: {title!r})" if title else ""))
        except Exception:
            pass
    return closed


def open_ztbv_table(s: SapSession, table: str, log=None) -> None:
    if log:
        log(f"SAP: /n{TRANSACTION} -> {table} @ plant {PLANT}")
    n_closed = close_lingering_modals(s, log=log)
    if n_closed and log:
        log(f"SAP: {n_closed} lingering modal(s) closed before navigation")
    s.find("wnd[0]").maximize()
    s.find("wnd[0]/tbar[0]/okcd").Text = f"/n{TRANSACTION}"
    s.find("wnd[0]").sendVKey(0)
    time.sleep(0.3)
    s.find("wnd[0]/usr/txtD_WERKS").Text = PLANT
    s.find("wnd[0]/usr/ctxtD_TAB").Text = table
    s.find("wnd[0]/usr/ctxtD_TAB").SetFocus()
    s.find("wnd[0]/usr/ctxtD_TAB").caretPosition = len(table)
    s.find("wnd[0]/tbar[1]/btn[8]").press()  # F8 -> selection screen
    time.sleep(0.3)


def paste_multi_value_filter(
    s: SapSession, push_button_id: str, values: list[str], log=None
) -> None:
    if log:
        log(f"SAP: pasting {len(values)} filter values via clipboard")
    s.find(push_button_id).press()
    time.sleep(0.3)
    # btn[24] = Upload from Clipboard, btn[8] = OK
    s.find("wnd[1]/tbar[0]/btn[24]").press()
    time.sleep(0.3)
    s.find("wnd[1]/tbar[0]/btn[8]").press()
    time.sleep(0.2)


def execute_query(s: SapSession, log=None) -> None:
    if log:
        log("SAP: executing query (F8)")
    s.find("wnd[0]/tbar[1]/btn[8]").press()
    time.sleep(0.5)


def export_alv_to_file(
    s: SapSession, target_dir: str, filename: str, log=None, timeout_s: int = 30
) -> str:
    """Try `&XXL` first, then fall back to `&PC`. Only accept files whose
    mtime is newer than the moment we triggered the export, so we can never
    return a prior step's data."""
    os.makedirs(target_dir, exist_ok=True)
    target_path = os.path.join(target_dir, filename)
    if os.path.exists(target_path):
        try:
            os.remove(target_path)
        except OSError:
            pass
    export_started_at = time.time()

    grid = s.find("wnd[0]/shellcont/shell")

    last_err: Exception | None = None
    for approach in ("XXL", "PC"):
        try:
            if approach == "XXL":
                if log:
                    log("SAP: exporting via &MB_EXPORT / &XXL")
                grid.pressToolbarContextButton("&MB_EXPORT")
                time.sleep(0.2)
                grid.selectContextMenuItem("&XXL")
            else:
                if log:
                    log("SAP: retrying export via &PC")
                grid.pressToolbarContextButton("&MB_EXPORT")
                time.sleep(0.2)
                grid.selectContextMenuItem("&PC")
                time.sleep(0.4)
                try:
                    s.find(
                        "wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150/"
                        "radSPOPLI-SELFLAG[1,0]"
                    ).Select()
                except Exception:
                    pass
                try:
                    s.find("wnd[1]/tbar[0]/btn[0]").press()
                except Exception:
                    pass
            time.sleep(0.5)
            s.find("wnd[1]/usr/ctxtDY_PATH").Text = target_dir
            s.find("wnd[1]/usr/ctxtDY_FILENAME").Text = filename
            s.find("wnd[1]/tbar[0]/btn[11]").press()
            deadline = time.time() + timeout_s
            while time.time() < deadline:
                if os.path.exists(target_path) and os.path.getsize(target_path) > 0:
                    try:
                        fresh = os.path.getmtime(target_path) >= export_started_at
                    except OSError:
                        fresh = False
                    if fresh:
                        if log:
                            log(f"SAP: export saved -> {target_path}")
                        return target_path
                time.sleep(0.25)
            raise SapError(f"Export timed out (>{timeout_s}s) waiting for {target_path}")
        except Exception as e:
            last_err = e
            if log:
                log(f"SAP: {approach} export attempt failed: "
                    f"{type(e).__name__}: {e}")
            try:
                s.find("wnd[1]/tbar[0]/btn[12]").press()  # Cancel
                if log:
                    log("SAP: cancelled leftover modal to prepare fallback")
            except Exception:
                pass
            time.sleep(0.3)
            continue

    raise SapError(
        "ALV export failed via both &XXL and &PC. Record one export via the "
        "SAP GUI Script Recorder and adjust the code."
    ) from last_err


# Multi-value push button IDs on the ZTBV selection screen per (table, field).
PUSH_BUTTONS = {
    ("LTAP", "TO_NUMBER"): "wnd[0]/usr/btn%_S3_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_CRNT", "RSNUM"): "wnd[0]/usr/btn%_S15_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_CRNT", "QMNUM"): "wnd[0]/usr/btn%_S29_%_APP_%-VALU_PUSH",
    ("Z50CFG_ENG_VALD", "OBJNR"): "wnd[0]/usr/btn%_S2_%_APP_%-VALU_PUSH",
}


## Pipeline core

Column-letter helpers, the SAP-result lookup builder, per-step and per-run
orchestration. Prints progress/log/status via a simple event handler
instead of the tkinter callback used by the .py app.


In [ ]:
def _col_letter(idx: int) -> str:
    result = ""
    n = idx
    while n > 0:
        n, r = divmod(n - 1, 26)
        result = chr(65 + r) + result
    return result


def _range(col_start: int, col_end: int, row_start: int, row_end: int) -> str:
    return f"{_col_letter(col_start)}{row_start}:{_col_letter(col_end)}{row_end}"


def _build_lookup(
    df: pd.DataFrame,
    key_cols: list[str],
    value_cols: list[str],
) -> tuple[dict, int, int]:
    """Return (lookup, dup_count, blank_key_count).

    - Duplicate composite keys keep the FIRST occurrence (dup_count++).
    - Rows where ANY key part is blank are skipped (blank_key_count++).
    """
    needed = list(dict.fromkeys(key_cols + value_cols))
    resolved: dict[str, str] = {}
    missing: list[str] = []
    for c in needed:
        r = _resolve_column(df, c)
        if r is None:
            missing.append(c)
        else:
            resolved[c] = r
    if missing:
        raise RuntimeError(
            "SAP export is missing these expected columns: "
            f"{missing}\n"
            f"Columns present in the export: {list(df.columns)}\n"
            "Fix: edit the ALV layout in ZTBV so each missing field is shown "
            "(prefer 'Technical Name' as the column title) and re-save the "
            "default variant, OR add another alias to SAP_COLUMN_ALIASES."
        )
    out: dict[str, dict] = {}
    dup_count = 0
    blank_key_count = 0
    for _, row in df.iterrows():
        parts = [normalize_key(row[resolved[c]]) for c in key_cols]
        if any(p == "" for p in parts):
            blank_key_count += 1
            continue
        composite = "|".join(parts)
        if composite in out:
            dup_count += 1
            continue
        out[composite] = {
            c: ("" if pd.isna(row[resolved[c]]) else row[resolved[c]])
            for c in value_cols
        }
    return out, dup_count, blank_key_count


def print_event(kind: str, payload) -> None:
    """Simple stdout event handler for notebook use. Mirrors the (kind,
    payload) contract used by the .py app's tkinter callback."""
    if kind == "log":
        msg, level = payload
        prefix = {"ok": "[OK]  ", "warn": "[WARN]", "error": "[ERR] ",
                  "info": "      "}.get(level, "      ")
        print(f"{prefix} {msg}")
    elif kind == "status":
        print(f"...    {payload}")
    elif kind == "progress":
        pass  # too noisy for stdout
    elif kind == "done":
        ok = payload
        print(f"===== workflow {'succeeded' if ok else 'FAILED'} =====")


def _log(on_event, msg: str, level: str = "info") -> None:
    on_event("log", (msg, level))


def _status(on_event, msg: str) -> None:
    on_event("status", msg)


def _progress(on_event, frac: float) -> None:
    on_event("progress", max(0.0, min(1.0, frac)))


def _run_step(
    step: LookupStep,
    xl: ExcelCtx,
    sap: SapSession,
    tmp_dir: str,
    on_event,
    step_index: int,
    total_steps: int,
    stop: threading.Event,
) -> tuple[int, int]:
    """Run one lookup step. Returns (matched, unmatched) row counts."""

    def sub_progress(sub_i: int, sub_n: int = 7):
        overall = (step_index + sub_i / sub_n) / total_steps
        _progress(on_event, overall)

    step_started = time.time()
    if stop.is_set():
        raise Cancelled()

    _status(on_event, f"[{step_index + 1}/{total_steps}] {step.name}: reading Excel keys")
    _log(on_event, f"--- Step {step_index + 1}/{total_steps}: {step.name}  "
                    f"(table={step.sap_table}, key_cols={step.key_columns})")
    sheet = xl.sheet

    # Each step derives its OWN last_row from its own primary key column;
    # matches the notebook, which re-computes last_row per step.
    primary_col = step.key_columns[0]
    last_row = last_row_in_column(sheet, primary_col)
    if last_row < 2:
        _log(on_event,
             f"No data below the header in column {_col_letter(primary_col)} "
             f"(#{primary_col}); step skipped", "warn")
        _progress(on_event, (step_index + 1) / total_steps)
        return (0, 0)

    key_col_min, key_col_max = min(step.key_columns), max(step.key_columns)
    key_range = _range(key_col_min, key_col_max, 2, last_row)
    key_rows = read_range_2d(sheet, key_range)
    sub_progress(1)

    first_key_offset = step.key_columns[0] - key_col_min

    def row_key(vals: list) -> str:
        offset_map = {c - key_col_min: c for c in step.key_columns}
        parts = [normalize_key(vals[offset]) for offset in sorted(offset_map)]
        return step.key_joiner.join(parts)

    excel_keys = [row_key(r) for r in key_rows]
    skip_flags = [_is_skip_key_cell(r[first_key_offset]) for r in key_rows]

    unique_paste_values: list[str] = []
    seen: set[str] = set()
    for r, skipped in zip(key_rows, skip_flags):
        if skipped:
            continue
        v = clean_numeric_for_sap(r[first_key_offset])
        if v and v not in seen:
            seen.add(v)
            unique_paste_values.append(v)

    n_skipped = sum(skip_flags)
    if n_skipped:
        _log(on_event,
             f"note: {n_skipped} row(s) in column {_col_letter(primary_col)} "
             f"are blank or 'NOT FOUND' -- not sent to SAP; their output "
             f"cells will be cleared just like any other non-match")

    if not unique_paste_values:
        _log(on_event,
             f"No usable values in column(s) {step.key_columns} "
             f"(after skipping blanks / 'NOT FOUND'); step is a no-op",
             "warn")
        _progress(on_event, (step_index + 1) / total_steps)
        return (0, 0)

    _log(on_event, f"{len(unique_paste_values)} unique key(s) will be sent to SAP")
    sub_progress(2)

    scratch = stage_values_on_clipboard(xl.app, unique_paste_values)
    try:
        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: loading {step.sap_table}")
        open_ztbv_table(sap, step.sap_table, log=lambda m: _log(on_event, m))
        sub_progress(3)

        push_id = PUSH_BUTTONS[(step.sap_table, step.push_button_field)]
        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: pasting {len(unique_paste_values)} filter values")
        paste_multi_value_filter(sap, push_id, unique_paste_values, log=lambda m: _log(on_event, m))
        sub_progress(4)

        _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: executing query")
        execute_query(sap, log=lambda m: _log(on_event, m))
        sub_progress(5)
    finally:
        close_scratch(scratch)

    if stop.is_set():
        raise Cancelled()

    _status(on_event, f"[{step_index + 1}/{total_steps}] SAP: exporting ALV grid")
    ts = int(time.time())
    export_name = f"{step.sap_table}_{step.push_button_field}_{ts}.xlsx"
    export_start = time.time()
    export_path = export_alv_to_file(
        sap, tmp_dir, export_name, log=lambda m: _log(on_event, m)
    )
    try:
        size_kb = os.path.getsize(export_path) / 1024.0
    except OSError:
        size_kb = 0.0
    _log(on_event, f"export finished in {time.time() - export_start:.1f}s "
                    f"({size_kb:.0f} KB)")
    sub_progress(6)

    _status(on_event, f"[{step_index + 1}/{total_steps}] loading SAP export")
    df = pd.read_excel(export_path)
    _log(on_event, f"SAP returned {len(df)} row(s) with columns {list(df.columns)[:8]}{'...' if len(df.columns) > 8 else ''}")
    extra_sap_cols = [e.sap_col for e in step.extras if e.sap_col]
    lookup, dup_count, blank_key_count = _build_lookup(
        df,
        step.sap_key_columns,
        step.sap_output_columns + extra_sap_cols,
    )
    if dup_count:
        _log(on_event,
             f"WARNING: SAP returned {dup_count} duplicate key(s); only the "
             f"FIRST row per key is used. Inspect the export or tighten your "
             f"ALV filter.", "warn")
    if blank_key_count:
        _log(on_event,
             f"note: skipped {blank_key_count} SAP row(s) with blank/partial "
             f"key columns", "info")

    _status(on_event, f"[{step_index + 1}/{total_steps}] matching keys and writing back to Excel")
    matched = 0
    entries_by_row: list[dict | None] = []
    for k, skipped in zip(excel_keys, skip_flags):
        if skipped:
            entries_by_row.append(None)
            continue
        entry = lookup.get(k)
        entries_by_row.append(entry)
        if entry is not None:
            matched += 1

    n_rows = len(excel_keys)
    rows_count = int(sheet.Rows.Count)

    with bulk_write(xl.app):
        # -------- Main output block ------------------------------------
        oc_min = min(step.excel_output_columns)
        oc_max = max(step.excel_output_columns)
        oc_span = oc_max - oc_min + 1
        target_write = _range(oc_min, oc_max, 2, last_row)
        # Clear the ENTIRE column range down to Rows.Count so stale rows
        # below last_row (from a prior, longer run) are wiped -- matches
        # notebook.
        target_clear = _range(oc_min, oc_max, 2, rows_count)

        main_out: list[list] = []
        for i in range(n_rows):
            entry = entries_by_row[i]
            row_vals = ["" for _ in range(oc_span)]
            if entry is not None:
                for j, sap_c in enumerate(step.sap_output_columns):
                    excel_c = step.excel_output_columns[j]
                    offset = excel_c - oc_min
                    val = entry[sap_c]
                    row_vals[offset] = "" if val is None else val
            main_out.append(row_vals)

        clear_range(sheet, target_clear)
        set_column_format_text(sheet, f"{_col_letter(oc_min)}:{_col_letter(oc_max)}")
        write_range_2d(sheet, target_write, main_out)

        # -------- Extras (per-column, per-column semantics) ------------
        for extra in step.extras:
            letter = _col_letter(extra.excel_col)
            xrange = f"{letter}2:{letter}{last_row}"

            need_existing = extra.preserve_on_nonmatch or extra.sap_col is None
            existing_extra = read_range_2d(sheet, xrange) if need_existing else None

            col_data: list[list] = []
            for i in range(n_rows):
                existing_val = (
                    existing_extra[i][0]
                    if (existing_extra and i < len(existing_extra)
                        and existing_extra[i])
                    else ""
                )
                entry = entries_by_row[i]
                if entry is not None:
                    if extra.sap_col is None:
                        # preserve on match (TO-3 J behavior)
                        col_data.append([existing_val])
                    else:
                        v = entry[extra.sap_col]
                        col_data.append(["" if v is None else v])
                else:
                    # non-match OR skip
                    if extra.preserve_on_nonmatch:
                        col_data.append([existing_val])
                    else:
                        col_data.append([""])

            # Only set text format on columns where we actually write SAP data;
            # a preserve-only column (sap_col=None) should keep whatever
            # NumberFormat the user had.
            if extra.sap_col is not None:
                set_column_format_text(sheet, f"{letter}:{letter}")
            write_range_2d(sheet, xrange, col_data)

        # -------- Match key column (P for TO-2) -------------------------
        if step.match_key_column is not None:
            mk_letter = _col_letter(step.match_key_column)
            sheet.Cells(1, step.match_key_column).Value = step.match_key_header
            set_column_format_text(sheet, f"{mk_letter}:{mk_letter}")
            mk_range = f"{mk_letter}2:{mk_letter}{last_row}"
            mk_data = [[k] for k in excel_keys]
            write_range_2d(sheet, mk_range, mk_data)

    sub_progress(7)

    non_skip = n_rows - n_skipped
    unmatched = non_skip - matched
    if len(unique_paste_values) > 20 and len(lookup) < max(1, len(unique_paste_values) // 10):
        _log(on_event,
             f"WARNING: sent {len(unique_paste_values)} unique key(s) to SAP but "
             f"only {len(lookup)} matched. Common causes: (a) query returned an "
             f"error screen, (b) the ALV filter rejected the values, (c) plant "
             f"or table wrong for this environment.", "warn")
    _log(on_event,
         f"Step {step_index + 1} finished in {time.time() - step_started:.1f}s: "
         f"{matched}/{non_skip} rows matched, {unmatched} unmatched"
         + (f", {n_skipped} skipped" if n_skipped else ""),
         "ok")
    _progress(on_event, (step_index + 1) / total_steps)
    return matched, unmatched


@dataclass
class RunConfig:
    excel_path: str
    workflow: str            # "TO" or "NOTIF"
    stop_event: threading.Event


def run(cfg: RunConfig, on_event) -> bool:
    """Entry point. Returns True on success."""
    pythoncom.CoInitialize()
    run_started = time.time()
    try:
        _log(on_event, f"esa-lookup starting workflow '{cfg.workflow}'", "info")

        _status(on_event, "opening Excel")
        xl = excel_attach(cfg.excel_path)
        _log(on_event, f"attached to Excel: {os.path.basename(cfg.excel_path)}", "ok")

        _status(on_event, "attaching to SAP GUI")
        sap = sap_attach()
        _log(on_event, "attached to SAP GUI session", "ok")

        steps = WORKFLOWS[cfg.workflow]
        total_steps = len(steps)
        tmp_dir = os.path.join(tempfile.gettempdir(), "esa_lookup")

        primary_col = steps[0].key_columns[0]
        primary_last = last_row_in_column(xl.sheet, primary_col)
        if primary_last < 2:
            _log(on_event,
                 f"No data below the header in column "
                 f"{_col_letter(primary_col)} (#{primary_col}) -- the "
                 f"workflow's primary input. Aborting.", "error")
            on_event("done", False)
            return False
        _log(on_event,
             f"step 1 will process rows 2..{primary_last} (column "
             f"{_col_letter(primary_col)}); each subsequent step derives "
             f"its own row range from its own key column")

        totals_matched = 0
        totals_seen = 0
        for i, step in enumerate(steps):
            if cfg.stop_event.is_set():
                raise Cancelled()
            m, u = _run_step(
                step, xl, sap, tmp_dir, on_event, i, total_steps,
                cfg.stop_event,
            )
            totals_matched += m
            totals_seen += m + u

        try:
            excel_save(xl.book)
        except ExcelError as e:
            _log(on_event, f"WARNING: {e}", "warn")

        _status(on_event, "done")
        _log(on_event, f"all steps complete: {totals_matched}/{totals_seen} row-matches across {total_steps} step(s)", "ok")
        _progress(on_event, 1.0)
        on_event("done", True)
        return True

    except Cancelled:
        _log(on_event, "cancelled by user", "warn")
        on_event("done", False)
        return False
    except SapError as e:
        _log(on_event, f"SAP error: {e}", "error")
        on_event("done", False)
        return False
    except ExcelError as e:
        _log(on_event, f"Excel error: {e}", "error")
        on_event("done", False)
        return False
    except Exception as e:
        _log(on_event, f"unexpected error: {e}", "error")
        _log(on_event, traceback.format_exc(), "error")
        on_event("done", False)
        return False
    finally:
        _log(on_event, f"total elapsed: {time.time() - run_started:.1f}s", "info")
        try:
            pythoncom.CoUninitialize()
        except Exception:
            pass


---

## Configure your run

Set the two variables below, then run the next cell.

- `EXCEL_PATH` — absolute path to your target workbook. Open it in Excel
  first (or leave closed and let the script open it).
- `WORKFLOW` — `"TO"` for the TO Number process (3 steps) or `"NOTIF"` for
  the Notification Number process (2 steps).


In [ ]:
# ---- EDIT THESE -----------------------------------------------------------
EXCEL_PATH = r"C:\path\to\your\workbook.xlsx"
WORKFLOW = "TO"   # "TO" or "NOTIF"
# ---------------------------------------------------------------------------


## Execute


In [ ]:
cfg = RunConfig(
    excel_path=EXCEL_PATH,
    workflow=WORKFLOW,
    stop_event=threading.Event(),
)
run(cfg, on_event=print_event)
